In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_secondary_structure_propensity)=
# Get secondary structure propensity

*Reading weighted Levitt reference propensities for selected amino-acid groups.*

Read relative frequencies from Levitt (1978), Tables III–V. This experimental
reference lookup uses group names. It neither predicts from sequence nor assigns
secondary structure from coordinates. A ratio above one is valid; the categories
are not probabilities and need not sum to one.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_secondary_structure_propensity` for arguments,
return values and diagnostics.
:::


## Basic usage

Let's show how this function works with the bundled pentalanine topology.
Select its five amino-acid groups explicitly: the terminal caps are outside the
reference table. No coordinates are needed.

In [2]:
import molsysmt as msm

In [3]:
molsys = msm.convert(
    msm.systems['pentalanine']['traj_pentalanine.h5msm'],
    to_form='molsysmt.Topology',
)
group_indices = [1, 2, 3, 4, 5]
helix = msm.physchem.get_secondary_structure_propensity(
    molsys, 'alpha_helix', selection=group_indices,
)
helix.tolist()

[1.29, 1.29, 1.29, 1.29, 1.29]

:::{admonition} Demo Systems Catalog
:class: dropdown note

See {ref}`Demo Systems <user-foundations-entrance-demo-systems>` for the bundled systems.
:::

## Choosing a reference category

Request `alpha_helix`, `beta_sheet` or `reverse_turn` explicitly. The single
supported definition is `levitt_1978_weighted`. These are weighted frequency
ratios from detailed Tables III, IV and V, respectively. Do not infer a `coil`
value by subtracting the other categories from one.

In [4]:
categories = ['alpha_helix', 'beta_sheet', 'reverse_turn']
{category: msm.physchem.get_secondary_structure_propensity(
    molsys, category, selection=[1], definition='levitt_1978_weighted',
).tolist() for category in categories}

{'alpha_helix': [1.29], 'beta_sheet': [0.9], 'reverse_turn': [0.77]}

## Preserving selection indices

Numeric group indices retain order and repetitions. Strings use `syntax`,
which defaults to `MolSysMT`; external languages need their optional provider.
Inputs may be native objects, dictionaries or files that supply group names.

In [5]:
reordered = msm.physchem.get_secondary_structure_propensity(
    molsys, 'alpha_helix', selection=[3, 1, 3],
)
selected = msm.physchem.get_secondary_structure_propensity(
    molsys, 'beta_sheet', selection='group_index in [1, 3]',
    syntax='MolSysMT',
)
reordered.tolist(), selected.tolist()

([1.29, 1.29, 1.29], [0.9, 0.9])

## Keeping dimensionless outputs

The detached float64 array has one entry per selected group. Empty selections
retain shape `(0,)`. These dimensionless ratios keep their values when the
application changes its length-unit policy; they are not areas or angles.

In [6]:
with msm.pyunitwizard.context(standard_units=['angstrom', 'ps', 'radians']):
    ratios = msm.physchem.get_secondary_structure_propensity(
        molsys, 'alpha_helix', selection=[1],
    )
empty = msm.physchem.get_secondary_structure_propensity(
    molsys, 'reverse_turn', selection=[],
)
ratios.tolist(), str(ratios.dtype), empty.shape

([1.29], 'float64', (0,))

## Understanding the scope

Only the twenty standard amino-acid names are supported. Modified groups,
terminal caps, DUM/X and unknown names fail explicitly. Names are lookup keys,
not chemical validation of a group's atoms. An atom subset does not redefine a
whole-group reference descriptor. Only `element='group'` is supported; no peptide
sum, class assignment or automatic average is defined.

Use {func}`molsysmt.structure.get_secondary_structure` for observed structural
assignment, which has different input requirements and method semantics.

## Recognizing the reference

The source is [Levitt (1978)](https://doi.org/10.1021/bi00613a026),
*Conformational preferences of amino acids in globular proteins*.
The existing sixty values were checked against the detailed original tables and
AAindex1 LEVM780101–103. The reverse-turn vector follows Table V; several
entries differ from summary Table II. Optional Ackredit credits the reference
scale for successful calls, including empty selections, without claiming to
execute the original population analysis.

:::{admonition} Related Tools & References
:class: dropdown seealso

- {func}`molsysmt.basic.convert`: supplying a topology in a supported form.
- {func}`molsysmt.physchem.get_secondary_structure_propensity`: reading reference ratios.
- {func}`molsysmt.structure.get_secondary_structure`: assigning observed secondary structure.
- {ref}`Cookbook_Export_Secondary_Structure_Propensities`: exporting named reference columns.
:::
